In [1]:
from pyspark.sql import SparkSession

spark = (
    SparkSession.builder
    .appName("Lab2-Transactions")
    .getOrCreate()
)
spark.sparkContext.setLogLevel("WARN")
print(f"Spark {spark.version} — gotowy")

Spark 4.0.0-preview2 — gotowy


In [3]:
df = spark.read.json("transactions_10k.jsonl")

print(f"Liczba rekordów: {df.count()}")
df.printSchema()

Liczba rekordów: 10000
root
 |-- amount: double (nullable = true)
 |-- category: string (nullable = true)
 |-- store: string (nullable = true)
 |-- timestamp: string (nullable = true)
 |-- tx_id: string (nullable = true)
 |-- user_id: string (nullable = true)



In [4]:
from pyspark.sql.functions import to_timestamp, col

df = df.withColumn("timestamp", to_timestamp(col("timestamp"), "yyyy-MM-dd HH:mm:ss"))

df.printSchema() 

root
 |-- amount: double (nullable = true)
 |-- category: string (nullable = true)
 |-- store: string (nullable = true)
 |-- timestamp: timestamp (nullable = true)
 |-- tx_id: string (nullable = true)
 |-- user_id: string (nullable = true)



## Z. 2.2

In [6]:
from pyspark.sql.functions import count, sum as _sum, avg, round as _round, min as _min, max as _max

store_summary = (
    df.groupBy("category")
    .agg(
        count("tx_id").alias("liczba_tx"),
        _round(_sum("amount"), 2).alias("suma_PLN"),
        _round(_min("amount"), 2).alias("minimalna_PLN"),
        _round(_max("amount"), 2).alias("maksymalna_PLN"),
    )
    .orderBy("category")
)
store_summary.show()

+-----------+---------+----------+-------------+--------------+
|   category|liczba_tx|  suma_PLN|minimalna_PLN|maksymalna_PLN|
+-----------+---------+----------+-------------+--------------+
|elektronika|     2542|1520770.69|          9.0|        9999.0|
|    książki|     2574| 851382.08|          5.0|       9107.25|
|     odzież|     2453| 849877.55|          5.0|       9696.63|
|    żywność|     2431| 789514.43|          5.0|       6916.92|
+-----------+---------+----------+-------------+--------------+



## Z. 3.2

In [15]:
from pyspark.sql.functions import window

window_store = (
    df.groupBy(window("timestamp", "30 minutes"), "store")
    .agg(
        count("tx_id").alias("liczba_tx"),
        _round(_sum("amount"), 2).alias("suma_PLN"),
    )
    .orderBy("window")
)
window_store.show(truncate=False)

+------------------------------------------+--------+---------+---------+
|window                                    |store   |liczba_tx|suma_PLN |
+------------------------------------------+--------+---------+---------+
|{2026-04-12 08:00:00, 2026-04-12 08:30:00}|Wrocław |296      |111540.59|
|{2026-04-12 08:00:00, 2026-04-12 08:30:00}|Gdańsk  |252      |93391.22 |
|{2026-04-12 08:00:00, 2026-04-12 08:30:00}|Kraków  |289      |117786.42|
|{2026-04-12 08:00:00, 2026-04-12 08:30:00}|Warszawa|275      |88441.58 |
|{2026-04-12 08:30:00, 2026-04-12 09:00:00}|Gdańsk  |514      |209187.85|
|{2026-04-12 08:30:00, 2026-04-12 09:00:00}|Wrocław |502      |215587.17|
|{2026-04-12 08:30:00, 2026-04-12 09:00:00}|Kraków  |532      |223541.41|
|{2026-04-12 08:30:00, 2026-04-12 09:00:00}|Warszawa|490      |182435.06|
|{2026-04-12 09:00:00, 2026-04-12 09:30:00}|Kraków  |590      |224358.03|
|{2026-04-12 09:00:00, 2026-04-12 09:30:00}|Warszawa|584      |214573.66|
|{2026-04-12 09:00:00, 2026-04-12 09:3

## Z 3.3

In [26]:
from pyspark.sql.functions import desc

krak_df = df[df['store'] == 'Kraków']

window_krk = (
    krak_df.groupBy(window("timestamp", "1 hour"))
    .agg(
        count("tx_id").alias("liczba_tx"),
        _round(_sum("amount"), 2).alias("suma_PLN"),
    )
    .orderBy(desc("suma_PLN"))
)
window_krk.show(truncate=False)

+------------------------------------------+---------+---------+
|window                                    |liczba_tx|suma_PLN |
+------------------------------------------+---------+---------+
|{2026-04-12 09:00:00, 2026-04-12 10:00:00}|1169     |483309.86|
|{2026-04-12 08:00:00, 2026-04-12 09:00:00}|821      |341327.83|
|{2026-04-12 10:00:00, 2026-04-12 11:00:00}|532      |201259.26|
+------------------------------------------+---------+---------+



## Z 4.2

In [27]:
tumbling_rows = (
    df.groupBy(window("timestamp", "1 hour"))
    .agg(count("tx_id"))
    .count()
)
sliding_rows = (
    df.groupBy(window("timestamp", "1 hour", "30 minutes"))
    .agg(count("tx_id"))
    .count()
)
print(f"Tumbling (1h):          {tumbling_rows} okien")
print(f"Sliding  (1h / 30min):  {sliding_rows} okien")

# Odpowiedz w komentarzu: dlaczego sliding ma więcej wierszy?
# TWOJA ODPOWIEDŹ: W przypadku zastosowania tumbling window, każde kolejne okno zaczynamy co jedną godzinę, a w przypadku sliding co pół godziny.

Tumbling (1h):          3 okien
Sliding  (1h / 30min):  7 okien


## Część 5

In [28]:
from pyspark.sql.functions import window

hourly = (
    df.groupBy(window("timestamp", "1 hour"))    # okno 1-godzinne
    .agg(
        count("tx_id").alias("liczba_tx"),
        _round(_sum("amount"), 2).alias("suma_PLN"),
    )
    .orderBy("window")
)
hourly.show(truncate=False)

+------------------------------------------+---------+----------+
|window                                    |liczba_tx|suma_PLN  |
+------------------------------------------+---------+----------+
|{2026-04-12 08:00:00, 2026-04-12 09:00:00}|3150     |1241911.3 |
|{2026-04-12 09:00:00, 2026-04-12 10:00:00}|4661     |1896230.21|
|{2026-04-12 10:00:00, 2026-04-12 11:00:00}|2189     |873403.24 |
+------------------------------------------+---------+----------+



In [29]:
# Odpowiedz na pytania w komentarzach:

# 1. Ile transakcji jest w oknie 09:00–10:00?
#    Sprawdź w wyniku zadania 3.1.
#    ODPOWIEDŹ: Jest 4661 transkacji.

# 2. Jaka jest różnica między groupBy("store") a groupBy(window(...), "store")?
#    ODPOWIEDŹ: W pierwszym przypadku grupujemy po sklepach po całym okresie. W drugim przypadku najpierw mamy grupowanie na zdefinoiwane przez nas okna, a dopiero w drugim kroku po sklepach.

# 3. W oknie sliding 1h/30min — ile okien zawiera transakcje z godziny 09:30?
#    Wskazówka: narysuj oś czasu.
#    ODPOWIEDŹ: Dwa okna zwierają transakcje z 09:30.

## Praca domowa 

Znajdź godzinę, w której sklep Gdańsk miał najniższą średnią kwotę transakcji.

In [37]:
gd_df = df[df['store'] == 'Gdańsk']

window_gd = (
    gd_df.groupBy(window("timestamp", "1 hour"))
    .agg(
        count("tx_id").alias("liczba_tx"),
        _round(avg("amount"), 2).alias("srednia_PLN"),
    )
    .orderBy(desc("srednia_PLN"))
)
window_gd.show(truncate=False)

+------------------------------------------+---------+-----------+
|window                                    |liczba_tx|srednia_PLN|
+------------------------------------------+---------+-----------+
|{2026-04-12 09:00:00, 2026-04-12 10:00:00}|1174     |415.91     |
|{2026-04-12 10:00:00, 2026-04-12 11:00:00}|558      |412.92     |
|{2026-04-12 08:00:00, 2026-04-12 09:00:00}|766      |395.01     |
+------------------------------------------+---------+-----------+



In [38]:
# O godzinie 8

Policz ile transakcji per kategoria było w oknie 09:00–09:30.

In [44]:
window_cat = (
    df.groupBy(window("timestamp", "30 minutes"), "category")
    .agg(
        count("tx_id").alias("liczba_tx"),
        _round(avg("amount"), 2).alias("srednia_PLN"),
    )
    .orderBy(desc("window"))
)

window_cat.show(truncate = False)

+------------------------------------------+-----------+---------+-----------+
|window                                    |category   |liczba_tx|srednia_PLN|
+------------------------------------------+-----------+---------+-----------+
|{2026-04-12 10:30:00, 2026-04-12 11:00:00}|elektronika|165      |549.16     |
|{2026-04-12 10:30:00, 2026-04-12 11:00:00}|książki    |208      |311.39     |
|{2026-04-12 10:30:00, 2026-04-12 11:00:00}|odzież     |164      |401.83     |
|{2026-04-12 10:30:00, 2026-04-12 11:00:00}|żywność    |212      |322.78     |
|{2026-04-12 10:00:00, 2026-04-12 10:30:00}|żywność    |344      |360.34     |
|{2026-04-12 10:00:00, 2026-04-12 10:30:00}|książki    |363      |340.78     |
|{2026-04-12 10:00:00, 2026-04-12 10:30:00}|odzież     |343      |325.09     |
|{2026-04-12 10:00:00, 2026-04-12 10:30:00}|elektronika|390      |575.72     |
|{2026-04-12 09:30:00, 2026-04-12 10:00:00}|żywność    |549      |346.97     |
|{2026-04-12 09:30:00, 2026-04-12 10:00:00}|odzież  

elektronika  -> 611  <br>
odzież      -> 605  <br>
książki     -> 622  <br>
żywność     -> 567 <br>

Zrób okno 15-minutowe i sprawdź w której ćwierćgodzinie był szczyt transakcji (łącznie dla wszystkich sklepów).

In [46]:
window_cat = (
    df.groupBy(window("timestamp", "15 minutes"), )
    .agg(
        count("tx_id").alias("liczba_tx"),
        _round(avg("amount"), 2).alias("srednia_PLN"),
        _round(_sum("amount"), 2).alias("suma_PLN"),
    )
    .orderBy(desc("suma_PLN"))
)

window_cat.show(truncate = False)

+------------------------------------------+---------+-----------+---------+
|window                                    |liczba_tx|srednia_PLN|suma_PLN |
+------------------------------------------+---------+-----------+---------+
|{2026-04-12 09:30:00, 2026-04-12 09:45:00}|1156     |436.8      |504943.74|
|{2026-04-12 09:15:00, 2026-04-12 09:30:00}|1234     |390.25     |481566.97|
|{2026-04-12 08:45:00, 2026-04-12 09:00:00}|1139     |417.25     |475251.18|
|{2026-04-12 09:45:00, 2026-04-12 10:00:00}|1100     |426.37     |469004.36|
|{2026-04-12 09:00:00, 2026-04-12 09:15:00}|1171     |376.36     |440715.14|
|{2026-04-12 10:00:00, 2026-04-12 10:15:00}|858      |418.71     |359254.89|
|{2026-04-12 08:30:00, 2026-04-12 08:45:00}|899      |395.44     |355500.31|
|{2026-04-12 10:15:00, 2026-04-12 10:30:00}|582      |385.63     |224438.4 |
|{2026-04-12 08:15:00, 2026-04-12 08:30:00}|644      |330.84     |213061.19|
|{2026-04-12 08:00:00, 2026-04-12 08:15:00}|468      |423.29     |198098.62|

In [47]:
# Miedzy 09:30-09:45 wystąpił szczyt sprzedaży 